# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by thinking about what I know about parrots. They\'re birds, right? Some of them, like parrots and parakeets, can mimic human speech. I remember seeing a parrot on a YouTube video that could say its owner\'s name. But why do they do that?\n\nFirst, maybe it\'s a learned behavior. Like, they imitate sounds because they\'re trying to communicate or bond with humans. I think some birds use mimicry to interact with their environment. But how does that translate to talking? Do they talk to get attention or to show they\'re part of their human\'s group?\n\nWait, in the wild, parrots live in flocks. They might use vocalizations to communicate with each other. So, when they\'re with humans, maybe they mimic human speech in the same way they would mimic other birds\' calls. It\'s a form of social bonding. If a parrot is with a human, it might try to imitate the human\'s voice to fit in.\n\nAnother angle: ma

# Using Tool

Either Bind the tool or Create the agent

## Binding tool

In [6]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the weather of a city"""
    return f"Weather of {city} is sunny"


model_with_tools = model.bind_tools([get_weather])

In [12]:
response = model_with_tools.invoke("What is weather like in Austin")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")

Tool: get_weather
Args: {'city': 'Austin'}


In [21]:
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Austin. Let me check the tools available. There\'s a function called get_weather that takes a city parameter. Since the user mentioned Austin, I need to call that function with "Austin" as the city argument. I should make sure the JSON is correctly formatted with the city name in the parameters. Alright, that should get the current weather information for Austin.\n', 'tool_calls': [{'id': 's9268ka9k', 'function': {'arguments': '{"city":"Austin"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 152, 'total_tokens': 258, 'completion_time': 0.180874976, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.013273104, 'prompt_tokens_details': None, 'queue_time': 0.225697335, 'total_time': 0.19414808}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 

## Create Agent

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model = "gpt-4o-mini",
    tools = [get_weather],
    system_prompt = "You are a helpful assistant."
)

# Tool Execution Loop

In [27]:
# Step 1 : Model generate tool call

messages = [{
    "role": "user",
    "content": "What is weather like in Austin"
}]

print(messages)

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

print(ai_msg)
print(messages)


[{'role': 'user', 'content': 'What is weather like in Austin'}]
content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Austin. Let me check the tools available. There\'s a function called get_weather that takes a city parameter. Since the user mentioned Austin, I need to call that function with "Austin" as the city argument. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'pp0tk0b4w', 'function': {'arguments': '{"city":"Austin"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 152, 'total_tokens': 249, 'completion_time': 0.133634544, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.006601884, 'prompt_tokens_details': None, 'queue_time': 0.105866296, 'total_time': 0.140236428}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tie

In [28]:
# Step 2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with generate areguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

print(tool_result)
print(messages)

content='Weather of Austin is sunny' name='get_weather' tool_call_id='pp0tk0b4w'
[{'role': 'user', 'content': 'What is weather like in Austin'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Austin. Let me check the tools available. There\'s a function called get_weather that takes a city parameter. Since the user mentioned Austin, I need to call that function with "Austin" as the city argument. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'pp0tk0b4w', 'function': {'arguments': '{"city":"Austin"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 152, 'total_tokens': 249, 'completion_time': 0.133634544, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.006601884, 'prompt_tokens_details': None, 'queue_time': 0.105866296, 'total_time': 0.14

In [25]:
# Step 3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Austin is currently sunny. Perfect day to enjoy outdoor activities! 😊
